## 1. Imports

In [5]:
import os

print(os.getcwd())

/content


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

from datasets import Dataset

from functions import (
    load_splits,
    evaluate,
    compute_metrics,
    LABELS,
    RANDOM_STATE,
)

ModuleNotFoundError: No module named 'functions'

## 2. Load data the canonical split

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = load_splits()

scores = []

print(len(X_train), len(X_val), len(X_test))

24238 5194 5194


## 3. Label encoding

In [ ]:
label2id = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2,
}

id2label = {v: k for k, v in label2id.items()}

y_train = y_train.map(label2id)
y_val = y_val.map(label2id)
y_test = y_test.map(label2id)

In [ ]:
print(y_train.value_counts().sort_index())

sentiment
0      569
1     1049
2    22620
Name: count, dtype: int64


## 4. Tokenization
###   4.1 Load tokenizer

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

###   4.2 Review length analysis

In [ ]:
sample = X_train.iloc[0]

print(sample)

Amazon fire 7 tablet. This is a good tablet, next time I buy a tablet I will get a bigger one. It does all we need it to. Great for kids!


In [ ]:
encoding = tokenizer(sample)

encoding.keys()

dict_keys(['input_ids', 'attention_mask'])

In [ ]:
print(encoding["input_ids"][:20])

[101, 9733, 2543, 1021, 13855, 1012, 2023, 2003, 1037, 2204, 13855, 1010, 2279, 2051, 1045, 4965, 1037, 13855, 1045, 2097]


In [ ]:
review_lengths = X_train.str.split().str.len()

review_lengths.describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99])

count    24238.000000
mean        33.819952
std         34.611427
min          2.000000
50%         24.000000
75%         39.000000
90%         63.000000
95%         84.000000
99%        160.630000
max       1868.000000
Name: text_full, dtype: float64

###  4.3 Maximum sequence length

The review length distribution was analysed before tokenization. Since 95% of the reviews contain fewer than 84 words (99% fewer than 161 words), a maximum sequence length of 128 tokens was selected. This provides a good balance between computational efficiency and information retention while reducing memory usage and training time.

In [ ]:
# 4.3 Maximum sequence length
MAX_LENGTH = 128

In [ ]:
# 4.4 Create Hugging Face datasets
train_ds = Dataset.from_dict({
    "text": X_train.tolist(),
    "label": y_train.tolist()
})

val_ds = Dataset.from_dict({
    "text": X_val.tolist(),
    "label": y_val.tolist()
})

test_ds = Dataset.from_dict({
    "text": X_test.tolist(),
    "label": y_test.tolist()
})

In [ ]:
# 4.5 Tokenization
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

###    4.4 Create Hugging Face datasets

In [ ]:
train_ds = Dataset.from_dict({
    "text": X_train.tolist(),
    "label": y_train.tolist()
})

val_ds = Dataset.from_dict({
    "text": X_val.tolist(),
    "label": y_val.tolist()
})

test_ds = Dataset.from_dict({
    "text": X_test.tolist(),
    "label": y_test.tolist()
})

In [ ]:
train_ds[0]

{'text': 'Amazon fire 7 tablet. This is a good tablet, next time I buy a tablet I will get a bigger one. It does all we need it to. Great for kids!',
 'label': 1}

###  4.5 Tokenize datasets

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

In [ ]:
train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Map:   0%|          | 0/24238 [00:00<?, ? examples/s]

Map:   0%|          | 0/5194 [00:00<?, ? examples/s]

Map:   0%|          | 0/5194 [00:00<?, ? examples/s]

###     4.6 Convert to PyTorch

In [ ]:
train_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

val_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

###   4.7 Verify tokenization

In [ ]:
sample = train_ds[0]

print(sample.keys())

dict_keys(['label', 'input_ids', 'attention_mask'])


In [ ]:
print(sample["input_ids"].shape)
print(sample["attention_mask"].shape)

torch.Size([128])
torch.Size([128])


## 5. Build DistilBERT model

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 6. Fine-tuning

### 6.1 Training hyperparameters

The following hyperparameters were selected as a standard starting point for fine-tuning DistilBERT on the sentiment classification task.

| Parameter | Value | Reason |
|------------|:-----:|--------|
| Number of epochs | **3** | Standard starting point for fine-tuning BERT-based models. |
| Learning rate | **2e-5** | One of the recommended learning rates in the original BERT paper; allows stable fine-tuning without large weight updates. |
| Batch size | **16** | Provides a good balance between training speed, memory usage, and model stability. |
| Weight decay | **0.01** | Helps reduce overfitting by applying L2 regularization during training. |
| Best model selection | **Validation loss** | Retains the checkpoint with the lowest validation loss rather than simply keeping the final epoch. |

In [ ]:
training_args = TrainingArguments(
    output_dir="./distilbert_results",

    evaluation_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    logging_steps=100,

    seed=RANDOM_STATE
)

/Users/felipemartignon/anaconda3/lib/python3.11/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


### 6.2 Create the Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

### 6.3 Start training

In [ ]:
trainer.train()

  0%|          | 0/4545 [00:00<?, ?it/s]

{'loss': 0.344, 'grad_norm': 1.8527209758758545, 'learning_rate': 1.955995599559956e-05, 'epoch': 0.07}
{'loss': 0.1956, 'grad_norm': 1.8619623184204102, 'learning_rate': 1.911991199119912e-05, 'epoch': 0.13}
{'loss': 0.2036, 'grad_norm': 8.524711608886719, 'learning_rate': 1.867986798679868e-05, 'epoch': 0.2}
{'loss': 0.1858, 'grad_norm': 1.3322519063949585, 'learning_rate': 1.8239823982398242e-05, 'epoch': 0.26}


KeyboardInterrupt: 

## 7. Validation

## 8. Final evaluation on the test set

## 9. Comparison with the baseline